<a href="https://colab.research.google.com/github/komazawa-deep-learning/komazawa-deep-learning.github.io/blob/master/2026notebooks/2026_1002addition_rnn_pytorch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# エンコーダ・デコーダモデル (seq2seq) による 足し算のデモ（PyTorch 版）

* 入力: "535+61"
* 出力 "596"
* 埋め草文字として空白の繰り返しを用いる

* 入力はオプションで反転させることが可能。
* 翻訳モデルなどでは，入力系列を反転させることで，性能向上が認められることが知られている。

## 文献

1. [Learning to Execute](http://arxiv.org/abs/1410.4615)
2. [Sequence to Sequence Learning with Neural Networks](http://papers.nips.cc/paper/5346-sequence-to-sequence-learning-with-neural-networks.pdf)

## 結果

理論的には、ソースとターゲットの間に短期的な依存関係が導入される。

* 2 桁 反転 足し算
    + 1 層 LSTM 99% 正解率

*  3 桁 反転 足し算
    + 1 層 LSTM 99% 正解率

* 4 桁 反転 足し算。
    + 1 層 LSTM 99% 正解率

* 5 桁 反転。
    + 1 層 LSTM 99%

## 構造上の特徴

- エンコーダ RNN で入力系列を読み込み、最終時刻の隠れ状態のみを取り出す（`return_sequences=False` 相当）
- その隠れ状態を `RepeatVector` で `DIGITS+1` 回複製し、デコーダ RNN の **各時刻に同じベクトルを入力** する
  （デコーダ自身の 1 つ前の出力を次の入力として使う、通常の自己回帰的 seq2seqとは異なる設計）
- デコーダの各時刻の出力に対して独立に全結合層 (`TimeDistributed(Dense)` 相当)と softmax を適用する

In [1]:
import numpy as np
import torch
import torch.nn as nn
import time

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'torch.__version__: {torch.__version__}')
print(f'device: {device}')

torch.__version__: 2.11.0+cpu
device: cpu


## CharacterTable

文字のワンホット符号化・復号化を行うクラス。

In [3]:
class CharacterTable(object):
    """任意の文字集合が問題として与えられた時に
    1. ワンホット 整数表現に符号化
    2. ワンホット 整数表現をその文字出力に復号化
    3. 出力として得られた確率ベクトルを，対応する文字に復号化
    """
    def __init__(self, chars):
        """文字表の初期化

        引数
            chars: 入力信号に現れる全文字集合
        """
        self.chars = sorted(set(chars))
        self.char_indices = dict((c, i) for i, c in enumerate(self.chars))
        self.indices_char = dict((i, c) for i, c in enumerate(self.chars))

    def encode(self, C, num_rows):
        """文字列 C のワンホット表現

        引数
            num_rows: ワンホット表現ベクトルを返す行数
        """
        x = np.zeros((num_rows, len(self.chars)))
        for i, c in enumerate(C):
            x[i, self.char_indices[c]] = 1
        return x

    def decode(self, x, calc_argmax=True):
        if calc_argmax:
            x = x.argmax(axis=-1)
        return ''.join(self.indices_char[xx] for xx in x)


In [4]:
class colors:
    ok = '\033[92m'    # 正解表示用の色を指定
    fail = '\033[91m'  # 失敗表示用の色を指定
    close = '\033[0m'  # 表示色をデフォルトに戻す

# 訓練に用いるパラメータ
TRAINING_SIZE = 50000
DIGITS = 3
INVERT = True

# 入力の最大長は `数字+数字` で表す
MAXLEN = DIGITS + 1 + DIGITS

# 入力に用いる文字と埋め草文字(空白)を指定
chars = '0123456789+ '
ctable = CharacterTable(chars)


In [5]:
questions = []
expected = []
seen = set()
print('データの生成...')
while len(questions) < TRAINING_SIZE:
    f = lambda: int(''.join(np.random.choice(list('0123456789'))
                    for i in range(np.random.randint(1, DIGITS + 1))))
    a, b = f(), f()

    # 既知の足し算問題をスキップ
    # 同様に x+Y == Y+x のような問題もスキップ。ソートもする。
    key = tuple(sorted((a, b)))
    if key in seen:
        continue
    seen.add(key)

    # 常にMAXLENになるようにデータを空白で埋める
    q = '{}+{}'.format(a, b)
    query = q + ' ' * (MAXLEN - len(q))
    ans = str(a + b)
    # 回答 `ans` は最大で DIGITS + 1 のサイズ
    ans += ' ' * (DIGITS + 1 - len(ans))
    if INVERT:
        # 例えば '12+345 ' は ' 543+21' となる。
        # 埋め草文字の空白注意
        query = query[::-1]
    questions.append(query)
    expected.append(ans)
print(f'総問題数: {len(questions)}')


データの生成...
総問題数: 50000


## ベクトル化


In [6]:
print('ベクトル化...')
x = np.zeros((len(questions), MAXLEN, len(chars)), dtype=np.float32)
y_idx = np.zeros((len(questions), DIGITS + 1), dtype=np.int64)
y_onehot = np.zeros((len(questions), DIGITS + 1, len(chars)), dtype=np.float32)
for i, sentence in enumerate(questions):
    x[i] = ctable.encode(sentence, MAXLEN)
for i, sentence in enumerate(expected):
    y_onehot[i] = ctable.encode(sentence, DIGITS + 1)
    for t, ch in enumerate(sentence):
        y_idx[i, t] = ctable.char_indices[ch]


ベクトル化...


In [7]:
# x の後半部分はほとんど大きな数字になるためデータ (x, y) をシャッフル
indices = np.arange(len(y_idx))
np.random.shuffle(indices)
x = x[indices]
y_idx = y_idx[indices]
y_onehot = y_onehot[indices]


In [8]:
# データセットの 10% を検証データセットとして設定。検証データセットでは学習を行わない
split_at = len(x) - len(x) // 10
(x_train, x_val) = x[:split_at], x[split_at:]
(y_train, y_val) = y_idx[:split_at], y_idx[split_at:]
y_val_onehot = y_onehot[split_at:]

print('訓練データセット数:')
print(f'入力データサイズ: {x_train.shape}')
print(f'教師データサイズ:{y_train.shape}')

print('検証データセット数:')
print(f'入力データサイズ: {x_val.shape}')
print(f'教師データサイズ: {y_val.shape}')

x_train_t = torch.from_numpy(x_train).to(device)
y_train_t = torch.from_numpy(y_train).to(device)
x_val_t = torch.from_numpy(x_val).to(device)
y_val_t = torch.from_numpy(y_val).to(device)


訓練データセット数:
入力データサイズ: (45000, 7, 12)
教師データサイズ:(45000, 4)
検証データセット数:
入力データサイズ: (5000, 7, 12)
教師データサイズ: (5000, 4)


In [9]:
# GRU や SimpleRNN(nn.RNN) に置き換えて実行してみること
RNN = nn.LSTM
RNN = nn.RNN
RNN = nn.GRU
HIDDEN_SIZE = 128
BATCH_SIZE = 128
LAYERS = 1


## モデル定義


In [10]:
class Seq2SeqAddition(nn.Module):
    """
    注意: デコーダの各時刻の入力として、エンコーダ最終隠れ状態を繰り返し与える設計、デコーダ自身の前時刻の出力を次の入力に使う自己回帰ではない。
    """
    def __init__(self, vocab_size, hidden_size, digits, rnn_cls=nn.LSTM, num_layers=1):
        super().__init__()
        self.digits = digits
        self.encoder = rnn_cls(vocab_size, hidden_size, num_layers=num_layers, batch_first=True)
        self.decoder = rnn_cls(hidden_size, hidden_size, num_layers=num_layers, batch_first=True)
        self.out = nn.Linear(hidden_size, vocab_size)
        self._is_lstm = (rnn_cls is nn.LSTM)

    def forward(self, x):
        if self._is_lstm:
            _, (h, c) = self.encoder(x)
        else:
            _, h = self.encoder(x)
        h_last = h[-1]  # 最終層・最終時刻の隠れ状態 (batch, hidden)
        repeated = h_last.unsqueeze(1).repeat(1, self.digits + 1, 1)  # RepeatVectorに相当
        dec_out, _ = self.decoder(repeated)
        logits = self.out(dec_out)  # (batch, digits+1, vocab) -- softmaxはCrossEntropyLoss内で計算
        return logits


print('モデル作成...')
model = Seq2SeqAddition(len(chars), HIDDEN_SIZE, DIGITS, rnn_cls=RNN, num_layers=LAYERS).to(device)
print(model)
n_params = sum(p.numel() for p in model.parameters())
print(f'パラメータ数: {n_params}')


モデル作成...
Seq2SeqAddition(
  (encoder): GRU(12, 128, batch_first=True)
  (decoder): GRU(128, 128, batch_first=True)
  (out): Linear(in_features=128, out_features=12, bias=True)
)
パラメータ数: 155148


## 正しさの検証

学習前に、出力 shape と勾配が正しく流れることを確認


In [11]:
_bx = x_train_t[:4]
_by = y_train_t[:4]
_logits = model(_bx)
assert _logits.shape == (4, DIGITS + 1, len(chars)), f'出力shape異常: {_logits.shape}'
_loss = nn.CrossEntropyLoss()(_logits.reshape(-1, _logits.size(-1)), _by.reshape(-1))
_loss.backward()
_has_grad = all(p.grad is not None for p in model.parameters())
assert _has_grad, '一部のパラメータに勾配が流れていない'
model.zero_grad()
print(f'shape OK: {_logits.shape}, loss OK: {_loss.item():.3f}, 勾配 OK')


shape OK: torch.Size([4, 4, 12]), loss OK: 2.492, 勾配 OK


## 学習

エポックごとに検証データからサンプルを取り出して予測結果を表示


In [12]:
optimizer = torch.optim.Adam(model.parameters())
criterion = nn.CrossEntropyLoss()

epochs = 30
n_sample = 10

for epoch in range(epochs):
    print()
    print('-' * 78)
    print(f'エポック数: {epoch + 1}')

    model.train()
    perm = torch.randperm(len(x_train_t))
    total_loss, total_correct_tok, total_tok = 0.0, 0, 0
    t0 = time.time()
    for i in range(0, len(perm), BATCH_SIZE):
        idx = perm[i:i + BATCH_SIZE]
        bx, by = x_train_t[idx], y_train_t[idx]
        optimizer.zero_grad()
        logits = model(bx)
        loss = criterion(logits.reshape(-1, logits.size(-1)), by.reshape(-1))
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * by.numel()
        total_correct_tok += (logits.argmax(-1) == by).sum().item()
        total_tok += by.numel()

    model.eval()
    with torch.no_grad():
        val_logits = model(x_val_t)
        val_loss = criterion(val_logits.reshape(-1, val_logits.size(-1)), y_val_t.reshape(-1)).item()
        val_pred = val_logits.argmax(-1)
        val_tok_acc = (val_pred == y_val_t).float().mean().item()
        val_seq_acc = (val_pred == y_val_t).all(dim=1).float().mean().item()
    elapsed = time.time() - t0
    print(f'loss: {total_loss/total_tok:.4f} - acc: {total_correct_tok/total_tok:.4f} - '
          f'val_loss: {val_loss:.4f} - val_acc(文字単位): {val_tok_acc:.4f} - '
          f'val_acc(系列完全一致): {val_seq_acc:.4f} - ({elapsed:.1f}秒)')

    # 検証データセットから n_sample 個の事例をランダムサンプリングして結果を表示
    model.eval()
    for i in range(n_sample):
        ind = np.random.randint(0, len(x_val))
        rowx_t = x_val_t[ind:ind + 1]
        with torch.no_grad():
            pred = model(rowx_t).argmax(-1)[0].cpu().numpy()
        q = ctable.decode(x_val[ind])
        correct = ctable.decode(y_val_onehot[ind])
        guess = ctable.decode(pred, calc_argmax=False)
        print('問:', q[::-1] if INVERT else q)
        print('答:', correct)
        if correct == guess:
            print(colors.ok + '正' + colors.close, end=" ")
        else:
            print(colors.fail + '誤' + colors.close, end=" ")
        print(guess)
        print('---')



------------------------------------------------------------------------------
エポック数: 1
loss: 1.8750 - acc: 0.3272 - val_loss: 1.7890 - val_acc(文字単位): 0.3410 - val_acc(系列完全一致): 0.0002 - (8.1秒)
問: 76+210 
答: 286 
誤 106 
---
問: 45+418 
答: 463 
誤 106 
---
問: 521+876
答: 1397
誤 106 
---
問: 293+867
答: 1160
誤 106 
---
問: 84+200 
答: 284 
誤 106 
---
問: 4+694  
答: 698 
誤 124 
---
問: 25+438 
答: 463 
誤 106 
---
問: 17+930 
答: 947 
誤 106 
---
問: 995+4  
答: 999 
誤 100 
---
問: 8+569  
答: 577 
誤 106 
---

------------------------------------------------------------------------------
エポック数: 2
loss: 1.7152 - acc: 0.3592 - val_loss: 1.6371 - val_acc(文字単位): 0.3928 - val_acc(系列完全一致): 0.0030 - (8.9秒)
問: 988+980
答: 1968
誤 1619
---
問: 662+10 
答: 672 
誤 685 
---
問: 953+61 
答: 1014
誤 100 
---
問: 31+709 
答: 740 
誤 389 
---
問: 954+4  
答: 958 
誤 100 
---
問: 371+45 
答: 416 
誤 482 
---
問: 22+653 
答: 675 
誤 322 
---
問: 679+202
答: 881 
誤 112 
---
問: 171+25 
答: 196 
誤 282 
---
問: 92+42  
答: 134 
誤 100 
---

-----------